# Nombre de validations entre 00h et 05h et ratio par rapport à la journée

In [70]:
import pandas as pd
from pathlib import Path

# =============================================================================
# PARAMÈTRES À ADAPTER
# =============================================================================

DOSSIER = Path(r"C:\Users\9920510N\SNCF\Datalab' Mass Transit GrpO365 - Documents\Rayonnement\Datathon\2026\Hackaviz IdFM\Données")
FEUILLE = 0                  # Première feuille Excel
FORMAT_DATE = None           # Exemple : "%d/%m/%Y"

COL_DATE = "JOUR"
COL_VALIDATIONS = "NB_VALD"
COL_CAT = "CAT_JOUR"
COL_TRANCHE = "TRNC_HORR_60"
COL_POURCENTAGE = "Pourcentage_validations"

# True : valeurs entre 0 et 100 ; False : valeurs entre 0 et 1
POURCENTAGE_SUR_100 = True

# Calendrier correspondant à tes données : dates incluses
PERIODES_VACANCES = [
    # ("2026-02-21", "2026-03-08"),
]

# Ajouter aussi les ponts si nécessaire
JOURS_FERIES = [
    # "2026-01-01",
]

FICHIERS = {
    "ferre": {
        "validations": "validations-reseau-ferre-nombre-validations-par-jour-1er-trimestre.xlsx",
        "profil": "validations-reseau-ferre-profils-horaires-par-jour-type-1er-trimestre.xlsx",
        "cle": "CODE_STIF_ARRET",
        "libelle": "LIBELLE_ARRET",
    },
    "surface": {
        "validations": "validations-reseau-surface-nombre-validations-par-jour-1er-trimestre.xlsx",
        "profil": "validation-reseau-surface-profils-horaires-par-jour-type-1er-trimestre.xlsx",
        "cle": "CODE_STIF_LIGNE",
        "libelle": "LIBELLE_LIGNE",
    },
}


# =============================================================================
# FONCTIONS
# =============================================================================

def lire_xlsx(nom_fichier):
    """Lit les données en conservant les codes sous forme de texte."""
    df = pd.read_excel(
        DOSSIER / nom_fichier,
        sheet_name=FEUILLE,
        dtype="string",
    )
    df.columns = df.columns.str.strip()
    return df


def determiner_cat_jour(date):
    """Classe chaque date dans une catégorie de jour."""
    if pd.isna(date):
        return pd.NA

    date = pd.Timestamp(date).normalize()
    feries = {pd.Timestamp(j).normalize() for j in JOURS_FERIES}

    if date in feries or date.weekday() == 6:
        return "DIJFP"

    en_vacances = any(
        pd.Timestamp(debut) <= date <= pd.Timestamp(fin)
        for debut, fin in PERIODES_VACANCES
    )

    if date.weekday() == 5:
        return "SAVS" if en_vacances else "SAHV"

    return "JOVS" if en_vacances else "JOHV"


def traiter_reseau(nom_reseau, config):
    cle = config["cle"]
    libelle = config["libelle"]

    # 1. Import des fichiers Excel
    validations = lire_xlsx(config["validations"])
    profil = lire_xlsx(config["profil"])

    # Vérification des colonnes nécessaires
    for df, colonnes, source in [
        (
            validations,
            [cle, libelle, COL_DATE, COL_VALIDATIONS],
            "validations",
        ),
        (
            profil,
            [cle, COL_CAT, COL_TRANCHE, COL_POURCENTAGE],
            "profil",
        ),
    ]:
        manquantes = [c for c in colonnes if c not in df.columns]
        if manquantes:
            raise ValueError(
                f"{nom_reseau} / {source} : colonnes absentes {manquantes}"
            )

    # 2. Nettoyage et filtrage JOHV en amont
    validations[cle] = validations[cle].astype("string").str.strip()
    profil[cle] = profil[cle].astype("string").str.strip()

    profil[COL_CAT] = profil[COL_CAT].astype("string").str.strip()
    profil[COL_TRANCHE] = profil[COL_TRANCHE].astype("string").str.strip()

    # Garder uniquement les profils JOHV
    profil = profil.loc[profil[COL_CAT] == "JOHV"].copy()

    # Convertir les dates et les validations
    validations[COL_DATE] = pd.to_datetime(
        validations[COL_DATE],
        format=FORMAT_DATE,
        dayfirst=True,
        errors="coerce",
    ).dt.normalize()

    validations[COL_VALIDATIONS] = pd.to_numeric(
        validations[COL_VALIDATIONS].astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )
    profil[COL_POURCENTAGE] = pd.to_numeric(
        profil[COL_POURCENTAGE].astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )

    validations = validations.dropna(
        subset=[cle, COL_DATE, COL_VALIDATIONS]
    ).copy()
    profil = profil.dropna(
        subset=[cle, COL_CAT, COL_TRANCHE, COL_POURCENTAGE]
    ).copy()

    # 3. Total des validations par code et par date
    journalieres = (
        validations
        .groupby([cle, COL_DATE], as_index=False)[COL_VALIDATIONS]
        .sum()
    )

    # 4. Ajouter CAT_JOUR puis ne conserver que les dates JOHV
    journalieres[COL_CAT] = "JOHV"

    # 5. Moyenne journalière par code pour les seules dates JOHV
    moyennes = (
        journalieres
        .groupby([cle, COL_CAT], as_index=False)
        .agg(
            NB_VALID_MOYEN=(COL_VALIDATIONS, "mean"),
            NB_JOURS=(COL_DATE, "nunique"),
        )
    )

    # 6. Préparation et jointure avec le profil horaire JOHV
    cles_profil = [cle, COL_CAT, COL_TRANCHE]

    profil = profil[
        cles_profil + [COL_POURCENTAGE]
    ].drop_duplicates(subset = cles_profil)

    
    if POURCENTAGE_SUR_100:
        profil["COEFF_HORAIRE"] = profil[COL_POURCENTAGE] / 100
    else:
        profil["COEFF_HORAIRE"] = profil[COL_POURCENTAGE]

    resultat = moyennes.merge(
        profil,
        on=[cle, COL_CAT],
        how="left",
        validate="one_to_many",
        indicator=True,
    )

    sans_profil = resultat["_merge"].eq("left_only").sum()
    if sans_profil:
        print(
            f"{nom_reseau} : {sans_profil} couple(s) code/JOHV "
            "sans profil horaire."
        )

    resultat = resultat.drop(columns="_merge")

    # 7. Calcul des validations horaires moyennes et sur la période
    resultat["NB_VALID_HORAIRE_MOYEN"] = (
        resultat["NB_VALID_MOYEN"] * resultat["COEFF_HORAIRE"]
    )
    resultat["NB_VALID_HORAIRE_TOTAL"] = (
        resultat["NB_VALID_HORAIRE_MOYEN"] * resultat["NB_JOURS"]
    )

    # Ajouter le libellé pour faciliter la lecture
    ref_libelles = (
        validations.groupby(cle, as_index=False)[libelle].first()
    )
    resultat = resultat.merge(
        ref_libelles,
        on=cle,
        how="left",
        validate="many_to_one",
    )

    return journalieres, moyennes, resultat


# =============================================================================
# EXÉCUTION POUR LES DEUX RÉSEAUX
# =============================================================================

resultats = {}

for nom_reseau, config in FICHIERS.items():
    journalieres, moyennes, jointure = traiter_reseau(
        nom_reseau, config
    )

    resultats[nom_reseau] = {
        "validations_journalieres": journalieres,
        "validations_moyennes_par_cat_jour": moyennes,
        "jointure_horaire": jointure,
    }

print("Traitement terminé.")

Traitement terminé.


In [72]:
display(resultats['surface']['jointure_horaire"].head(10))

,CODE_STIF_LIGNE,CAT_JOUR,NB_VALID_MOYEN,NB_JOURS,TRNC_HORR_60,Pourcentage_validations,COEFF_HORAIRE,NB_VALID_HORAIRE_MOYEN,NB_VALID_HORAIRE_TOTAL,LIBELLE_LIGNE
0,-,JOHV,755.083333,36,11H-12H,5.31,0.0531,40.094925,1443.4173,LIGNE NON DEFINIE
1,-,JOHV,755.083333,36,17H-18H,11.02,0.1102,83.210183,2995.5666,LIGNE NON DEFINIE
2,-,JOHV,755.083333,36,19H-20H,3.62,0.0362,27.334017,984.0246,LIGNE NON DEFINIE
3,-,JOHV,755.083333,36,21H-22H,1.32,0.0132,9.9671,358.8156,LIGNE NON DEFINIE
4,-,JOHV,755.083333,36,6H-7H,3.09,0.0309,23.332075,839.9547,LIGNE NON DEFINIE
5,-,JOHV,755.083333,36,12H-13H,5.74,0.0574,43.341783,1560.3042,LIGNE NON DEFINIE
6,-,JOHV,755.083333,36,1H-2H,0.02,0.0002,0.151017,5.4366,LIGNE NON DEFINIE
7,-,JOHV,755.083333,36,14H-15H,5.41,0.0541,40.850008,1470.6003,LIGNE NON DEFINIE
8,-,JOHV,755.083333,36,15H-16H,6.82,0.0682,51.496683,1853.8806,LIGNE NON DEFINIE
9,-,JOHV,755.083333,36,18H-19H,7.48,0.0748,56.480233,2033.2884,LIGNE NON DEFINIE


In [75]:
# Calcul du chiffre clé
#Validations entre 00h et 05h

heures_nuit = ["0H-1H", "1H-2H", "2H-3H", "3H-4H","4H-5H"]
val_total = resultats['surface']['jointure_horaire']["NB_VALID_HORAIRE_MOYEN"].sum() + resultats['ferre']['jointure_horaire']["NB_VALID_HORAIRE_MOYEN"].sum()

val_0_5_surface = resultats['surface']['jointure_horaire'][(resultats['surface']['jointure_horaire']['TRNC_HORR_60'].isin(heures_nuit))]
val_0_5_ferre = resultats['ferre']['jointure_horaire'][(resultats['ferre']['jointure_horaire']['TRNC_HORR_60'].isin(heures_nuit))]

val_nuit = val_0_5_surface["NB_VALID_HORAIRE_MOYEN"].sum() + val_0_5_ferre["NB_VALID_HORAIRE_MOYEN"].sum()



In [76]:
display(val_nuit/val_total)

np.float64(0.022554766566430292)

In [77]:
display(val_nuit)

np.float64(210697.38699254088)

# Nombre de bus Noctilien et nombre d'arrêts à 3h

In [32]:
FICHIER = r"C:\Users\9920510N\SNCF\Datalab' Mass Transit GrpO365 - Documents\Rayonnement\Datathon\2026\Hackaviz IdFM\Données\offre_hebdomadaire_moyenne_hors_vacances.csv" 
data_offre = pd.read_csv(FICHIER, sep=None, engine="python",encoding='utf-8')


In [33]:
display(data_offre.head(10))

,﻿code_mode_ligne,libelle_mode_ligne,id_ligne_commerciale,nom_ligne_commerciale,id_zone_de_correspondance,id_zone_arret,id_arret,nom_arret,tranche_horaire,nb_courses_lundi,...,nb_courses_jeudi,nb_courses_vendredi,nb_courses_samedi,nb_courses_dimanche,code_departement,code_commune,nom_commune,longitude_arret,latitude_arret,epci
0,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,8,4.0,...,4.0,4.0,1.0,1.000000,94.0,94077.0,Villeneuve-le-Roi,2.408523,48.741875,Métropole du Grand Paris
1,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,10,2.0,...,2.0,2.0,2.0,1.000000,94.0,94077.0,Villeneuve-le-Roi,2.408523,48.741875,Métropole du Grand Paris
2,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,22,1.0,...,1.0,1.0,1.0,0.000000,94.0,94077.0,Villeneuve-le-Roi,2.408523,48.741875,Métropole du Grand Paris
3,3,Bus,C00001,482,463511.0,463510,463512,Cimetières,10,2.0,...,2.0,2.0,2.0,1.000000,94.0,94077.0,Villeneuve-le-Roi,2.408657,48.742056,Métropole du Grand Paris
4,3,Bus,C00001,482,463511.0,463510,463512,Cimetières,20,2.0,...,2.0,2.0,1.0,1.000000,94.0,94077.0,Villeneuve-le-Roi,2.408657,48.742056,Métropole du Grand Paris
5,3,Bus,C00001,482,463515.0,463514,463513,Rue de la Paix,11,2.0,...,2.0,2.0,2.0,1.966667,94.0,94077.0,Villeneuve-le-Roi,2.429753,48.741798,Métropole du Grand Paris
6,3,Bus,C00001,482,463515.0,463514,463513,Rue de la Paix,12,2.0,...,2.0,2.0,2.0,1.033333,94.0,94077.0,Villeneuve-le-Roi,2.429753,48.741798,Métropole du Grand Paris
7,3,Bus,C00001,482,463515.0,463514,463513,Rue de la Paix,14,3.0,...,3.0,3.0,2.0,1.000000,94.0,94077.0,Villeneuve-le-Roi,2.429753,48.741798,Métropole du Grand Paris
8,3,Bus,C00001,482,463515.0,463514,463513,Rue de la Paix,15,2.0,...,2.0,2.0,2.0,1.000000,94.0,94077.0,Villeneuve-le-Roi,2.429753,48.741798,Métropole du Grand Paris
9,3,Bus,C00001,482,463515.0,463514,463513,Rue de la Paix,18,4.0,...,4.0,4.0,2.0,1.000000,94.0,94077.0,Villeneuve-le-Roi,2.429753,48.741798,Métropole du Grand Paris


In [102]:
df_bus_noctilien = data_offre[(data_offre.libelle_mode_ligne == "Bus") & (data_offre.nom_ligne_commerciale.str[0] == "N") & ((data_offre.nom_ligne_commerciale.str.len().isin([2,3,4])) | (data_offre.nom_ligne_commerciale=="N151 (ex N151)")) & (data_offre.tranche_horaire.isin([24,25,26,27,4]))]
#On ne garde que les lignes de bus commencant par N et ne dépassant pas 4 caractère pour rester dans la nomenclature Noctilien
#On se restreint également à la plage minuit-5h

In [98]:
display(nb_arret)

2897

In [99]:
display(nb_dessertes)

np.float64(25065.828571428574)

In [100]:
display(nb_dessertes_par_arret)

np.float64(8.652339858967405)

In [101]:
display(nb_dessertes_par_arret_par_min)

np.float64(34.6727018228573)